# Level 9: NISQ Era and Hybrid Quantum-Classical Algorithms
## Q# Edition

In this notebook, you will:

1. Understand NISQ limitations and **variational approaches**
2. Build a **parameterized quantum circuit** in Q#
3. Implement a simple **VQE** with classical optimization
4. Explore **Azure Quantum resource estimation**

In [ ]:
import qsharp
from qsharp_widgets import Circuit
print("Q# ready!")

---
## 1. Parameterized Circuits in Q#

VQE and QAOA rely on quantum circuits with tunable parameters. Q# handles this naturally with operation arguments.

In [ ]:
%%qsharp

open Microsoft.Quantum.Math;
open Microsoft.Quantum.Diagnostics;

// Parameterized ansatz for 2-qubit VQE
operation Ansatz(params : Double[], qs : Qubit[]) : Unit {
    // Layer 1: Ry rotations
    Ry(params[0], qs[0]);
    Ry(params[1], qs[1]);

    // Entangling layer
    CNOT(qs[0], qs[1]);

    // Layer 2: More Ry rotations
    Ry(params[2], qs[0]);
    Ry(params[3], qs[1]);
}

In [ ]:
%%qsharp

// Measure expectation value of ZZ operator via sampling
operation MeasureZZ(params : Double[], shots : Int) : Double {
    mutable total = 0.0;

    for _ in 0..shots - 1 {
        use qs = Qubit[2];
        Ansatz(params, qs);

        let r0 = M(qs[0]);
        let r1 = M(qs[1]);

        // ZZ eigenvalue: +1 if same, -1 if different
        let eigenvalue = (r0 == r1) ? 1.0 | -1.0;
        set total += eigenvalue;

        ResetAll(qs);
    }

    return total / IntAsDouble(shots);
}

// Measure expectation value of XI operator
operation MeasureXI(params : Double[], shots : Int) : Double {
    mutable total = 0.0;

    for _ in 0..shots - 1 {
        use qs = Qubit[2];
        Ansatz(params, qs);

        // Measure X on qubit 0: rotate to computational basis
        H(qs[0]);
        let r0 = M(qs[0]);
        let eigenvalue = r0 == Zero ? 1.0 | -1.0;
        set total += eigenvalue;

        ResetAll(qs);
    }

    return total / IntAsDouble(shots);
}

// Measure expectation value of IX operator
operation MeasureIX(params : Double[], shots : Int) : Double {
    mutable total = 0.0;

    for _ in 0..shots - 1 {
        use qs = Qubit[2];
        Ansatz(params, qs);

        H(qs[1]);
        let r1 = M(qs[1]);
        let eigenvalue = r1 == Zero ? 1.0 | -1.0;
        set total += eigenvalue;

        ResetAll(qs);
    }

    return total / IntAsDouble(shots);
}

// Full Hamiltonian: H = -1.0*ZZ + 0.5*XI + 0.5*IX
operation ComputeEnergy(params : Double[], shots : Int) : Double {
    let zz = MeasureZZ(params, shots);
    let xi = MeasureXI(params, shots);
    let ix = MeasureIX(params, shots);

    return -1.0 * zz + 0.5 * xi + 0.5 * ix;
}

In [ ]:
# Test the energy computation with random parameters
import numpy as np

test_params = list(np.random.random(4) * 2 * np.pi)
params_str = str(test_params).replace(' ', '')
energy = qsharp.eval(f"ComputeEnergy({params_str}, 2000)")
print(f"Energy with random params: {energy:.4f}")
print(f"Exact ground state energy: -1.2071")

In [ ]:
# VQE optimization loop using classical optimizer
from scipy.optimize import minimize
import matplotlib.pyplot as plt

energy_history = []

def vqe_objective(params):
    params_list = list(params)
    params_str = str(params_list).replace(' ', '')
    energy = qsharp.eval(f"ComputeEnergy({params_str}, 1000)")
    energy_history.append(energy)
    return energy

initial_params = np.random.random(4) * 2 * np.pi

print("Running VQE optimization...")
result = minimize(vqe_objective, initial_params, method='COBYLA',
                  options={'maxiter': 100})

print(f"\nVQE Result:")
print(f"  Optimized energy: {result.fun:.4f}")
print(f"  Exact energy:     -1.2071")
print(f"  Iterations:       {result.nfev}")

plt.figure(figsize=(10, 5))
plt.plot(energy_history, 'b-', alpha=0.7)
plt.axhline(y=-1.2071, color='r', linestyle='--', label='Exact ground state')
plt.xlabel('Iteration')
plt.ylabel('Energy')
plt.title('VQE Convergence (Q# + Classical Optimizer)')
plt.legend()
plt.show()

---
## 2. Resource Estimation

Q# and Azure Quantum provide resource estimation to understand what it takes to run algorithms on real hardware.

Key metrics:
- **Logical qubits**: How many error-corrected qubits
- **T gates**: The most expensive gates in error-corrected computing  
- **Physical qubits**: Total qubits needed including error correction overhead
- **Runtime**: Estimated wall-clock time

This helps answer: "Is this algorithm practical to run on near-future hardware?"

In [ ]:
%%qsharp

// A simple operation to estimate resources for
operation SampleAlgorithm() : Result {
    use qs = Qubit[4];
    
    // Some typical operations
    for q in qs { H(q); }
    CNOT(qs[0], qs[1]);
    CNOT(qs[1], qs[2]);
    CNOT(qs[2], qs[3]);
    
    for q in qs { T(q); }
    
    CNOT(qs[3], qs[2]);
    CNOT(qs[2], qs[1]);
    CNOT(qs[1], qs[0]);
    
    let result = M(qs[0]);
    ResetAll(qs);
    return result;
}

In [ ]:
# Get circuit info
circuit = qsharp.circuit("SampleAlgorithm()")
Circuit(circuit)

---
## 3. QAOA Preview in Q#

In [ ]:
%%qsharp

// Simple QAOA for MaxCut on a triangle (3 vertices)
operation QAOAMaxCut(gamma : Double, beta : Double) : Result[] {
    use qs = Qubit[3];

    // Initial superposition
    for q in qs { H(q); }

    // Problem Hamiltonian: ZZ interactions for each edge
    // Edge 0-1
    CNOT(qs[0], qs[1]);
    Rz(gamma, qs[1]);
    CNOT(qs[0], qs[1]);

    // Edge 0-2
    CNOT(qs[0], qs[2]);
    Rz(gamma, qs[2]);
    CNOT(qs[0], qs[2]);

    // Edge 1-2
    CNOT(qs[1], qs[2]);
    Rz(gamma, qs[2]);
    CNOT(qs[1], qs[2]);

    // Mixer
    for q in qs { Rx(2.0 * beta, q); }

    let results = MeasureEachZ(qs);
    ResetAll(qs);
    return results;
}

In [ ]:
# Run QAOA and count cut values
edges = [(0, 1), (0, 2), (1, 2)]
cuts = {0: 0, 1: 0, 2: 0, 3: 0}

for _ in range(2000):
    r = qsharp.eval("QAOAMaxCut(0.7854, 0.3927)")
    bits = [1 if x == 'One' else 0 for x in r]
    cut = sum(1 for (i, j) in edges if bits[i] != bits[j])
    cuts[cut] = cuts.get(cut, 0) + 1

print("QAOA MaxCut Results:")
for cut_val, count in sorted(cuts.items()):
    print(f"  Cut = {cut_val}: {count} times")

print(f"\nOptimal cut = 2 (putting 1 vertex alone)")

---
## 4. Key Takeaways

| Concept | Description |
|---------|-------------|
| **NISQ** | Current era: noisy, limited qubits |
| **VQE in Q#** | Parameterized operations + classical optimizer |
| **QAOA** | Combinatorial optimization via quantum mixing |
| **Resource estimation** | Plan for future fault-tolerant hardware |

---
## 5. Challenges

1. **Deeper ansatz**: Add a third rotation + entangling layer. Does VQE converge faster?
2. **QAOA depth 2**: Add a second QAOA layer (2 gamma, 2 beta parameters). Does it improve?
3. **Different Hamiltonian**: Change the Hamiltonian coefficients and re-run VQE.

In [ ]:
%%qsharp

// Your challenge code here!

---
**Next up: [Level 10 — Quantum Error Correction](../Level_10_Error_Correction/Level_10_QSharp.ipynb)**